In [1]:
%cd /content


/content


In [2]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [3]:
!ls /content/drive/MyDrive

 AI_Resume_Analyzer_and_Job_Recommendation_System
 AQVH.gsheet
'assignment 02-Feb-2024 12-44-42.pdf'
 Brain_Stroke
 Classroom
'Colab Notebooks'
'Document from dorarevanth67'
'Document from Revanth.pdf'
'Document from Tom And Jerry 🥰🥰'
 IMG20230820162749.jpg
 IMG20230820162752.jpg
 IMG20230820162805.jpg
 IMG20230820162809.jpg
 IMG20230820163455.jpg
 IMG20230820163553.jpg
 IMG20230820163616.jpg
'Kishore Bday Pics'
 kom
 N211086.jpg
'physics assignment 19-Aug-2022 21-58-29.pdf'
'Share fees puc1.pdf'
'Share fees puc2.pdf'
'Untitled spreadsheet.gsheet'
 VID20240104141034_2.mp4
'WhatsApp Chat with .... (1).txt'
'WhatsApp Chat with .....txt'
'Working progress.gdoc'
'wt project'


In [4]:
%cd /content/drive/MyDrive/AI_Resume_Analyzer_and_Job_Recommendation_System


/content/drive/MyDrive/AI_Resume_Analyzer_and_Job_Recommendation_System


In [5]:
!ls

data  notebooks  saved_models  src  tmp_trainer


In [6]:
!find data -type f

data/resume.zip
data/jobdescription.zip
data/extracted/Resume/Resume.csv
data/extracted/data/data/ACCOUNTANT/10554236.pdf
data/extracted/data/data/ACCOUNTANT/10674770.pdf
data/extracted/data/data/ACCOUNTANT/11163645.pdf
data/extracted/data/data/ACCOUNTANT/11759079.pdf
data/extracted/data/data/ACCOUNTANT/12065211.pdf
data/extracted/data/data/ACCOUNTANT/12202337.pdf
data/extracted/data/data/ACCOUNTANT/12338274.pdf
data/extracted/data/data/ACCOUNTANT/12442909.pdf
data/extracted/data/data/ACCOUNTANT/12780508.pdf
data/extracted/data/data/ACCOUNTANT/12802330.pdf
data/extracted/data/data/ACCOUNTANT/13072019.pdf
data/extracted/data/data/ACCOUNTANT/13130984.pdf
data/extracted/data/data/ACCOUNTANT/13294301.pdf
data/extracted/data/data/ACCOUNTANT/13491889.pdf
data/extracted/data/data/ACCOUNTANT/13701259.pdf
data/extracted/data/data/ACCOUNTANT/14055988.pdf
data/extracted/data/data/ACCOUNTANT/14126433.pdf
data/extracted/data/data/ACCOUNTANT/14224370.pdf
data/extracted/data/data/ACCOUNTANT/14449423.

In [7]:
!ls src/models


bert  distilbert  roberta  sentence_bert


In [8]:
!mkdir -p src/models/sentence_bert
!mkdir -p saved_models/sentence_bert

In [9]:
!ls sr

ls: cannot access 'sr': No such file or directory


In [10]:

!pip install sentence-transformers

In [11]:
!touch src/models/sentence_bert/train.py
!touch src/models/sentence_bert/generate_embeddings.py
!touch src/models/sentence_bert/matcher.py
!touch src/models/sentence_bert/evaluate.py

In [12]:
!ls src/models/sentence_bert


evaluate.py  generate_embeddings.py  matcher.py  train.py


In [13]:
!ls src/models/sentence_bert

evaluate.py  generate_embeddings.py  matcher.py  train.py


In [14]:
%%writefile src/models/sentence_bert/train.py

from sentence_transformers import SentenceTransformer


MODEL_NAME = "all-MiniLM-L6-v2"


def load_model():

    model = SentenceTransformer(MODEL_NAME)

    return model


if __name__ == "__main__":

    model = load_model()

    print("Sentence-BERT model loaded successfully")

Overwriting src/models/sentence_bert/train.py


In [15]:
!python src/models/sentence_bert/train.py

Loading weights: 100% 103/103 [00:00<00:00, 18125.17it/s]
Sentence-BERT model loaded successfully


In [16]:
%%writefile src/models/sentence_bert/generate_embeddings.py

import pandas as pd
import torch
import os

from sentence_transformers import SentenceTransformer


MODEL_NAME = "all-MiniLM-L6-v2"


def generate_embeddings():

    print("Loading job dataset...")

    df = pd.read_csv(
        "data/processed(job matcher)/processed_jobs.csv"
    )

    print("Dataset shape:", df.shape)


    print("Loading Sentence-BERT model...")

    model = SentenceTransformer(MODEL_NAME)


    texts = df["clean_job_text"].tolist()


    print("Generating embeddings...")


    embeddings = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_tensor=True
    )


    os.makedirs(
        "saved_models/sentence_bert",
        exist_ok=True
    )


    torch.save(
        embeddings,
        "saved_models/sentence_bert/job_embeddings.pt"
    )


    df.to_csv(
        "saved_models/sentence_bert/job_metadata.csv",
        index=False
    )


    print("Job embeddings saved successfully")


if __name__ == "__main__":

    generate_embeddings()

Overwriting src/models/sentence_bert/generate_embeddings.py


In [17]:
!python src/models/sentence_bert/generate_embeddings.py

Loading job dataset...
Dataset shape: (110898, 7)
Loading Sentence-BERT model...
Loading weights: 100% 103/103 [00:00<00:00, 13575.93it/s]
Generating embeddings...
Batches: 100% 3466/3466 [3:51:42<00:00,  4.01s/it]
Job embeddings saved successfully


In [18]:
%%writefile src/models/sentence_bert/matcher.py

import torch
import pandas as pd

from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim


MODEL_NAME = "all-MiniLM-L6-v2"


class JobMatcher:


    def __init__(self):

        print("Loading Sentence-BERT...")

        self.model = SentenceTransformer(
            MODEL_NAME
        )


        self.job_embeddings = torch.load(
            "saved_models/sentence_bert/job_embeddings.pt"
        )


        self.jobs = pd.read_csv(
            "saved_models/sentence_bert/job_metadata.csv"
        )


        print("Matcher ready")



    def recommend(
        self,
        resume_text,
        top_k=5
    ):


        resume_embedding = self.model.encode(
            resume_text,
            convert_to_tensor=True
        )


        scores = cos_sim(
            resume_embedding,
            self.job_embeddings
        )[0]


        top_results = torch.topk(
            scores,
            k=top_k
        )


        recommendations = []


        for score, index in zip(
            top_results.values,
            top_results.indices
        ):


            job = self.jobs.iloc[index.item()]


            recommendations.append({

                "job_title":
                    job["title"],

                "industry":
                    job["industry_name"],

                "match_score":
                    round(
                        score.item()*100,
                        2
                    )

            })


        return recommendations



if __name__ == "__main__":


    matcher = JobMatcher()


    resume = """
    Python developer with experience
    in machine learning, deep learning,
    NLP, SQL and data analysis
    """


    results = matcher.recommend(
        resume,
        top_k=5
    )


    print("\nRecommended Jobs\n")


    for job in results:

        print(job)

Overwriting src/models/sentence_bert/matcher.py


In [19]:
!python src/models/sentence_bert/matcher.py

Loading Sentence-BERT...
Loading weights: 100% 103/103 [00:00<00:00, 5060.78it/s]
Matcher ready

Recommended Jobs

{'job_title': 'Python Developer', 'industry': 'IT Services and IT Consulting', 'match_score': 70.27}
{'job_title': 'Python Developer', 'industry': 'Software Development', 'match_score': 70.2}
{'job_title': 'Lead Python Software Engineer', 'industry': 'IT Services and IT Consulting', 'match_score': 69.98}
{'job_title': 'Python Developer with Django, Flask', 'industry': 'IT Services and IT Consulting', 'match_score': 69.02}
{'job_title': 'Python Developer', 'industry': 'Technology, Information and Internet', 'match_score': 68.95}


In [20]:
%%writefile src/models/sentence_bert/evaluate.py

from matcher import JobMatcher



def evaluate():

    matcher = JobMatcher()


    resume = """

    Experienced software developer
    with Python, Java, SQL,
    machine learning and NLP skills.

    Worked on data analysis,
    deep learning and backend systems.

    """


    results = matcher.recommend(
        resume,
        top_k=5
    )


    print("\nTop Recommended Jobs\n")


    for i, job in enumerate(results, 1):

        print(
            f"{i}. {job}"
        )



if __name__ == "__main__":

    evaluate()

Overwriting src/models/sentence_bert/evaluate.py


In [21]:
!python src/models/sentence_bert/evaluate.py

Loading Sentence-BERT...
Loading weights: 100% 103/103 [00:00<00:00, 5381.13it/s]
Matcher ready

Top Recommended Jobs

1. {'job_title': 'Python Developer', 'industry': 'IT Services and IT Consulting', 'match_score': 72.8}
2. {'job_title': 'Data Science & Visualization Engineer', 'industry': 'IT Services and IT Consulting', 'match_score': 71.91}
3. {'job_title': 'Python Perl Developer', 'industry': 'Information Services', 'match_score': 71.38}
4. {'job_title': 'Python Developer with Django, Flask', 'industry': 'IT Services and IT Consulting', 'match_score': 71.0}
5. {'job_title': 'Java developer- python', 'industry': 'IT Services and IT Consulting', 'match_score': 71.0}


In [1]:
!ls saved_models/sentence_bert

ls: cannot access 'saved_models/sentence_bert': No such file or directory


In [2]:
%cd /content/drive/MyDrive/AI_Resume_Analyzer_and_Job_Recommendation_System

[Errno 2] No such file or directory: '/content/drive/MyDrive/AI_Resume_Analyzer_and_Job_Recommendation_System'
/content


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
!ls /content/drive/MyDrive

 AI_Resume_Analyzer_and_Job_Recommendation_System
 AQVH.gsheet
'assignment 02-Feb-2024 12-44-42.pdf'
 Brain_Stroke
 Classroom
'Colab Notebooks'
'Document from dorarevanth67'
'Document from Revanth.pdf'
'Document from Tom And Jerry 🥰🥰'
 IMG20230820162749.jpg
 IMG20230820162752.jpg
 IMG20230820162805.jpg
 IMG20230820162809.jpg
 IMG20230820163455.jpg
 IMG20230820163553.jpg
 IMG20230820163616.jpg
'Kishore Bday Pics'
 kom
 N211086.jpg
'physics assignment 19-Aug-2022 21-58-29.pdf'
'Share fees puc1.pdf'
'Share fees puc2.pdf'
'Untitled spreadsheet.gsheet'
 VID20240104141034_2.mp4
'WhatsApp Chat with .... (1).txt'
'WhatsApp Chat with .....txt'
'Working progress.gdoc'
'wt project'


In [5]:
%cd /content/drive/MyDrive/AI_Resume_Analyzer_and_Job_Recommendation_System

/content/drive/MyDrive/AI_Resume_Analyzer_and_Job_Recommendation_System


In [6]:
!ls

data  notebooks  saved_models  src  tmp_trainer


In [7]:
!ls saved_models/sentence_bert

job_embeddings.pt  job_metadata.csv


In [8]:
!pwd

/content/drive/MyDrive/AI_Resume_Analyzer_and_Job_Recommendation_System


In [9]:
!ls src/models/sentence_bert

evaluate.py  generate_embeddings.py  matcher.py  __pycache__  train.py
